# Lake polygons from binary mask GeoTIFFs (0/1)

This notebook:
1. Loops over **.tif/.tiff** mask files in a folder (pixel values 0/1; **1 = lake**)
2. Vectorizes lake pixels into polygons
3. Builds a **GeoDataFrame** with per-lake area
4. Classifies lakes by area and reports counts per class

> Notes:
- If your rasters are in a **projected CRS** (meters), areas are computed directly.
- If in a **geographic CRS** (degrees), the polygons are reprojected to an **appropriate UTM zone** (per file) before area calculation.


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from shapely.geometry import shape
from shapely.ops import unary_union

print('Versions:')
import shapely, pyproj
print('  geopandas:', gpd.__version__)
print('  rasterio :', rasterio.__version__)
print('  shapely :', shapely.__version__)
print('  pyproj  :', pyproj.__version__)


Versions:
  geopandas: 1.1.1
  rasterio : 1.4.3
  shapely : 2.1.1
  pyproj  : 3.7.2


In [2]:
# --------- USER SETTINGS ---------
# Folder containing your mask GeoTIFFs
MASK_DIR = r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\imageries_256\labels"   # <- change this

# Optional: set a minimum lake area filter (in square meters) to drop tiny artifacts
MIN_AREA_M2 = 0.0

# Lake size classes (km^2) - edit as needed
AREA_BINS_KM2 = [0, 0.01, 0.1, 1, 10, np.inf]
AREA_LABELS = [
    "<0.01",
    "0.01–0.1",
    "0.1–1",
    "1–10",
    "≥10",
]

# Output paths (optional)
OUT_GPKG = "lake_polygons.gpkg"   # saved in current working directory
OUT_LAYER = "lakes"
OUT_CSV = "lake_counts_by_class.csv"


In [3]:
def is_geographic_crs(crs) -> bool:
    """Return True if CRS is geographic (degrees)."""
    if crs is None:
        return False
    try:
        return bool(crs.is_geographic)
    except Exception:
        return False

def utm_epsg_from_lonlat(lon: float, lat: float) -> int:
    """Pick a UTM EPSG code from lon/lat."""
    zone = int((lon + 180) // 6) + 1
    # Northern hemisphere: 326xx, Southern: 327xx
    return 32600 + zone if lat >= 0 else 32700 + zone

def vectorize_mask_to_polygons(mask: np.ndarray, transform, lake_value: int = 1):
    """Vectorize mask pixels equal to lake_value into shapely polygons."""
    # Ensure boolean mask
    valid = (mask == lake_value)
    if valid.sum() == 0:
        return []
    # shapes yields (geom, value)
    geoms = []
    for geom, val in features.shapes(mask.astype(np.uint8), mask=valid, transform=transform):
        if val == lake_value:
            geoms.append(shape(geom))
    return geoms


In [4]:
mask_dir = Path(MASK_DIR)
tif_paths = sorted([p for p in mask_dir.glob('*.tif')] + [p for p in mask_dir.glob('*.tiff')])

if not tif_paths:
    raise FileNotFoundError(f"No .tif/.tiff files found in: {mask_dir}")

records = []

for tif_path in tif_paths:
    with rasterio.open(tif_path) as src:
        mask = src.read(1)  # first band
        transform = src.transform
        crs = src.crs
        geoms = vectorize_mask_to_polygons(mask, transform, lake_value=1)
        if not geoms:
            continue
        # Build a GDF for this file in raster CRS
        gdf_file = gpd.GeoDataFrame(
            {
                "source_file": [tif_path.name] * len(geoms),
                "geometry": geoms,
            },
            crs=crs,
        )

        # Compute area in meters^2:
        if crs is not None and not is_geographic_crs(crs):
            gdf_area = gdf_file.copy()
        else:
            # Reproject to UTM based on polygon centroid (approx) for area
            # If CRS missing, assume it's already projected (best-effort)
            if crs is None:
                gdf_area = gdf_file.copy()
            else:
                centroid = gdf_file.unary_union.centroid
                utm_epsg = utm_epsg_from_lonlat(centroid.x, centroid.y)
                gdf_area = gdf_file.to_crs(epsg=utm_epsg)

        gdf_area["area_m2"] = gdf_area.geometry.area
        gdf_area = gdf_area[gdf_area["area_m2"] >= float(MIN_AREA_M2)].copy()

        # Add sequential lake id within file
        gdf_area["lake_id"] = range(1, len(gdf_area) + 1)
        gdf_area["area_km2"] = gdf_area["area_m2"] / 1e6

        # Store back in original CRS for output consistency
        if gdf_file.crs is not None and gdf_area.crs != gdf_file.crs:
            gdf_out = gdf_area.to_crs(gdf_file.crs)
            # keep computed area fields (already computed in projected CRS)
            gdf_out["area_m2"] = gdf_area["area_m2"].values
            gdf_out["area_km2"] = gdf_area["area_km2"].values
            gdf_out["lake_id"] = gdf_area["lake_id"].values
        else:
            gdf_out = gdf_area

        records.append(gdf_out)

lakes_gdf = gpd.GeoDataFrame(pd.concat(records, ignore_index=True), crs=records[0].crs)
lakes_gdf.head()

C:\Users\gg\AppData\Local\Temp\ipykernel_26204\2213595499.py:35: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = gdf_file.unary_union.centroid
C:\Users\gg\AppData\Local\Temp\ipykernel_26204\2213595499.py:35: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = gdf_file.unary_union.centroid
C:\Users\gg\AppData\Local\Temp\ipykernel_26204\2213595499.py:35: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = gdf_file.unary_union.centroid
C:\Users\gg\AppData\Local\Temp\ipykernel_26204\2213595499.py:35: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = gdf_file.unary_union.centroid
C:\Users\gg\AppData\Local\Temp\ipykernel_26204\2213595499.py:35: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead

,source_file,geometry,area_m2,lake_id,area_km2
0,image_1.tif,"POLYGON ((74.05711 34.83354, 74.05711 34.83315...",466514.834887,1,0.466515
1,image_10.tif,"POLYGON ((74.06222 34.82741, 74.06222 34.82286...",214999.905691,1,0.215000
2,image_100.tif,"POLYGON ((73.3864 35.36144, 73.3864 35.36138, ...",6336.088791,1,0.006336
3,image_100.tif,"POLYGON ((73.39324 35.3568, 73.39324 35.35677,...",50069.500811,2,0.050070
4,image_100.tif,"POLYGON ((73.38703 35.3548, 73.38703 35.35476,...",25019.913940,3,0.025020


In [5]:
# Classify lakes by area and count
lakes_gdf["area_class_km2"] = pd.cut(
    lakes_gdf["area_km2"],
    bins=AREA_BINS_KM2,
    labels=AREA_LABELS,
    include_lowest=True,
    right=False,
)

counts = (
    lakes_gdf.groupby("area_class_km2", dropna=False)
    .size()
    .rename("lake_count")
    .reset_index()
)
counts

C:\Users\gg\AppData\Local\Temp\ipykernel_26204\558235484.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  lakes_gdf.groupby("area_class_km2", dropna=False)


,area_class_km2,lake_count
0,<0.01,2069
1,0.01–0.1,680
2,0.1–1,63
3,1–10,0
4,≥10,0


In [ ]:
# Optional: save outputs
# 1) GeoPackage (recommended)
lakes_gdf.to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")

# 2) CSV summary
counts.to_csv(OUT_CSV, index=False)

print('Saved:')
print(' ', os.path.abspath(OUT_GPKG))
print(' ', os.path.abspath(OUT_CSV))


## Tips
- If you see lots of tiny polygons, set `MIN_AREA_M2` (e.g., `MIN_AREA_M2 = 50` or `100`).
- If your masks have nodata values or extra classes, adjust `vectorize_mask_to_polygons(..., lake_value=1)`.
- If you want to merge touching lake pixels into a single feature per lake, `rasterio.features.shapes` already groups contiguous regions (same value) into one polygon.
